# forced_alignment

> Routes for toggling between NLTK and force-aligned pre-splits, plus rendering helpers for FA UI controls. Trigger and progress are handled by `cjm-fasthtml-job-monitor`.

In [ ]:
#| default_exp routes.forced_alignment

In [ ]:
#| export
from typing import Any, Dict, List, Tuple, Callable, Optional
from dataclasses import asdict

from fasthtml.common import Div, Span, Button, APIRouter

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_styles
from cjm_fasthtml_daisyui.components.layout.join import join, join_item
from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.utilities.typography import font_size
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items
from cjm_fasthtml_tailwind.core.base import combine_classes
from cjm_fasthtml_lucide_icons.factory import lucide_icon

from cjm_workflow_state.state_store import SQLiteWorkflowStateStore
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT
from cjm_transcript_segmentation.models import TextSegment, SegmentationUrls
from cjm_transcript_segmentation.routes.handlers import build_mutation_response
from cjm_transcript_vad_align.models import VADChunk

from cjm_transcript_segment_align.html_ids import CombinedHtmlIds
from cjm_transcript_segment_align.components.step_renderer import (
    render_alignment_status, render_seg_mini_stats_badge,
)

## HTML IDs for Forced Alignment UI

In [ ]:
#| export
# Stable ID for FA controls area within toolbar (progress polling target)
FA_SLOT_ID = CombinedHtmlIds.FA_TOOLBAR_SLOT

## UI Rendering Functions

The toggle renderer stays here (domain-specific). The trigger button and progress
indicator are now provided by `cjm-fasthtml-job-monitor`.

In [ ]:
#| export
def render_fa_toggle(
    active_presplit: str,  # "nltk" or "forced_alignment"
    toggle_url: str,  # URL for toggle route
) -> Any:  # Toggle button group
    """Render the NLTK / Force Aligned toggle button group."""
    nltk_active = active_presplit == "nltk"
    fa_active = active_presplit == "forced_alignment"

    nltk_cls = combine_classes(
        btn, btn_sizes.sm, join_item,
        btn_colors.primary if nltk_active else btn_styles.outline
    )
    fa_cls = combine_classes(
        btn, btn_sizes.sm, join_item,
        btn_colors.primary if fa_active else btn_styles.outline
    )

    return Div(
        Span("Pre-splits:", cls=combine_classes(font_size.sm, m.r(2))),
        Div(
            Button(
                lucide_icon("text-cursor-input", size=4, cls=str(m.r(1))),
                "NLTK",
                cls=nltk_cls,
                hx_post=toggle_url,
                hx_vals='{"mode": "nltk"}',
                hx_swap="none",
                disabled="disabled" if nltk_active else None,
            ),
            Button(
                lucide_icon("audio-waveform", size=4, cls=str(m.r(1))),
                "Force Aligned",
                cls=fa_cls,
                hx_post=toggle_url,
                hx_vals='{"mode": "forced_alignment"}',
                hx_swap="none",
                disabled="disabled" if fa_active else None,
            ),
            cls=join,
        ),
        cls=combine_classes(flex_display, items.center),
    )

## Route Handlers

In [ ]:
#| export
async def _handle_fa_toggle(
    state_store: SQLiteWorkflowStateStore,
    workflow_id: str,
    request: Any,
    sess: Any,
    seg_urls: SegmentationUrls,
    toggle_url: str,
) -> Any:  # Targeted OOB updates (slots + stats + toolbar with toggle + alignment status + mini-stats)
    """Toggle between NLTK and force-aligned pre-split snapshots.
    
    Loads the selected pre-split snapshot as the current working state.
    Pushes the previous working state to undo history (single history model).
    """
    from cjm_transcript_segment_align.components.handlers import segments_match_presplit
    from cjm_workflow_state.history import push_history

    form = await request.form()
    target_mode = form.get("mode", "nltk")

    session_id = get_session_id(sess)
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    seg_state = step_states.get("segmentation", {})
    align_state = step_states.get("alignment", {})

    nltk_presplit = seg_state.get("nltk_presplit", [])
    fa_presplit = seg_state.get("fa_presplit", [])
    current_segments = seg_state.get("segments", [])

    # Determine which snapshot to load
    if target_mode == "nltk":
        target_segments = nltk_presplit
    else:
        target_segments = fa_presplit

    # No-op if current already matches target
    if segments_match_presplit(current_segments, target_segments):
        return ()

    # Push current state to undo history before switching
    history = seg_state.get("history", [])
    focused_index = seg_state.get("focused_index", 0)
    snapshot = {"segments": current_segments, "focused_index": focused_index}
    history = push_history(history, snapshot, max_depth=20)

    # Load target snapshot as current working state
    seg_state["segments"] = target_segments[:]
    seg_state["initial_segments"] = target_segments[:]
    seg_state["focused_index"] = 0
    seg_state["history"] = history

    step_states["segmentation"] = seg_state
    state["step_states"] = step_states
    state_store.update_state(workflow_id, session_id, state)

    # Build FA toggle for toolbar extra_actions
    fa_toggle_content = Div(
        render_fa_toggle(target_mode, toggle_url),
        id=FA_SLOT_ID,
    )

    # NLTK Split disabled when target is NLTK
    nltk_disabled = (target_mode == "nltk")

    # Build targeted mutation response
    seg_oob = build_mutation_response(
        segment_dicts=target_segments,
        focused_index=0,
        visible_count=seg_state.get("visible_count", DEFAULT_VISIBLE_COUNT),
        history_depth=len(history),
        urls=seg_urls,
        extra_actions=fa_toggle_content,
        nltk_split_disabled=nltk_disabled,
    )

    # Cross-domain OOB
    segments = [TextSegment.from_dict(s) for s in target_segments]
    chunk_count = len(align_state.get("vad_chunks", []))
    alignment_status_oob = render_alignment_status(len(segments), chunk_count, oob=True)
    mini_stats_oob = render_seg_mini_stats_badge(segments, oob=True)

    return (*seg_oob, alignment_status_oob, mini_stats_oob)

## Router Assembly

In [ ]:
#| export
def init_forced_alignment_routers(
    state_store: SQLiteWorkflowStateStore,  # State store instance
    workflow_id: str,  # Workflow identifier
    seg_urls: SegmentationUrls,  # Segmentation URL bundle
    prefix: str,  # Route prefix (e.g., "/fa")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize forced alignment routes (toggle only).
    
    Trigger and progress are handled by `cjm-fasthtml-job-monitor`.
    This router provides the domain-specific toggle between NLTK/FA pre-splits.
    """
    router = APIRouter(prefix=prefix)

    @router
    async def toggle(request, sess):
        """Toggle between NLTK and force-aligned pre-splits."""
        return await _handle_fa_toggle(
            state_store, workflow_id,
            request, sess, seg_urls,
            toggle_url=toggle.to(),
        )

    routes = {
        "toggle": toggle,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()